# 🧰 Python Built-ins & Standard Library — The Power Tools

> **What you'll learn:** the built-in functions you'll use every day, iterators and lazy generators, `lambda`, `functools` (partial, `lru_cache`, `reduce`, `wraps`), closures and decorators, context managers, `collections`, `itertools`, dates and time zones, JSON, and `os`/`sys`/`pathlib` — then build a streaming log analyzer over a real Apache server log.

| | |
|---|---|
| **Difficulty** | 🟢 Beginner → 🔴 Interview level (decorators, closures, generators) |
| **Time** | ~4 hours to read and run, +2 hours for exercises and the project |
| **Prerequisites** | [Python Basics](01_Python_Basics.ipynb) (collections, loops, comprehensions, functions, exceptions) |
| **Tested with** | Python 3.12 (standard library only — nothing to install) |
| **Interview relevance** | ⭐⭐⭐ High — "write a decorator with arguments", "generator vs list", "implement an LRU cache", closure late binding, `groupby` bugs, time zones |

## 🤔 What Is the Standard Library?

Python ships "batteries included". Think of a new kitchen that already comes with knives, pans, and a timer:

- **Built-in functions** (`len`, `sorted`, `zip`, `enumerate`, …) are the knives on the counter — always there, no import needed.
- **Standard library modules** (`collections`, `itertools`, `functools`, `datetime`, `json`, `pathlib`, …) are the drawers — one `import` away, no installation.

This notebook also covers three **language features** that professionals use constantly:

| Feature | Plain-English idea | You've already used it in… |
|---|---|---|
| **Generator** | a function that hands out values *one at a time*, on demand | `range()`, reading a file line by line |
| **Decorator** | a wrapper that adds behaviour to a function without editing it | `@app.get("/")` in FastAPI, `@torch.no_grad()` |
| **Context manager** | guaranteed setup + cleanup around a block of code | `with open(...) as f:` |

## 🎯 Why It Matters

- **ML library APIs are built from these pieces.** PyTorch `DataLoader`s are iterators, Hugging Face `datasets` stream with generators, FastAPI routes and `pytest` fixtures are decorators, `torch.no_grad()` and MLflow runs are context managers.
- **Real data is too big for memory.** Generators let you process a 50 GB log or dataset line by line with almost no RAM.
- **Less code, fewer bugs.** `Counter`, `defaultdict`, and `itertools` replace dozens of hand-written loops with tested, fast building blocks.
- **Interviews love them.** "Write a retry decorator", "implement an LRU cache", "why does my generator return nothing the second time?", "what's wrong with this `groupby`?", and "how do you store timestamps?" are all common questions.

## ✅ By the End You Can

- [ ] Use `sorted`/`min`/`max` with `key`, `any`/`all`, `zip`, `enumerate`, `map`/`filter` idiomatically
- [ ] Explain iterables vs iterators, write generators, and measure the memory they save
- [ ] Write closures and decorators (including decorators with arguments) that preserve metadata with `functools.wraps`
- [ ] Write context managers as a class and with `@contextmanager` (with `try/finally`)
- [ ] Pick the right tool from `collections`, `itertools`, `functools`, `datetime`/`zoneinfo`, and `json`
- [ ] Build `map`, `zip`, `range`, an LRU-cache decorator, and a timer context manager from scratch

## 📋 Table of Contents

1. [Built-ins You Use Every Day](#1.-Built-ins-You-Use-Every-Day-🟢)
2. [Lambda, map, and filter](#2.-Lambda,-map,-and-filter-🟢)
3. [Iterators and the Iteration Protocol](#3.-Iterators-and-the-Iteration-Protocol-🟡)
4. [Generators and Lazy Pipelines](#4.-Generators-and-Lazy-Pipelines-🟡)
5. [functools: partial, reduce, lru_cache](#5.-functools:-partial,-reduce,-lru_cache-🟡)
6. [Closures](#6.-Closures-🟡)
7. [Decorators](#7.-Decorators-🔴)
8. [Context Managers](#8.-Context-Managers-🟡)
9. [collections: Counter, defaultdict, deque, namedtuple](#9.-collections:-Counter,-defaultdict,-deque,-namedtuple-🟢)
10. [itertools](#10.-itertools-🟡)
11. [Dates and Times: datetime and zoneinfo](#11.-Dates-and-Times:-datetime-and-zoneinfo-🟡)
12. [JSON](#12.-JSON-🟢)
13. [os, sys, and pathlib](#13.-os,-sys,-and-pathlib-🟢)
- [🔧 Build It From Scratch](#🔧-Build-It-From-Scratch) · [⚠️ Common Pitfalls](#⚠️-Common-Pitfalls) · [🏋️ Practice Exercises](#🏋️-Practice-Exercises) · [🚀 Mini Project](#🚀-Mini-Project:-Streaming-Apache-Log-Analyzer) · [🎤 Interview Q&A](#🎤-Interview-Q&A) · [🧪 Quick Quiz](#🧪-Quick-Quiz) · [📚 Resources](#📚-Resources) · [📝 Summary](#📝-Summary-Cheat-Sheet)

## ⚙️ Setup

Run the cell below first. Everything used here ships with Python — no installs.

The `check()` helper gives you instant feedback on exercises: **✅** correct, **⏳** not attempted yet, **❌** wrong (with a hint).

In [1]:
# Nothing to install: this notebook uses only Python's standard library (Python 3.12+ for itertools.batched).

import contextlib
import functools
import itertools
import json
import math
import os
import random
import sys
import time
import tracemalloc
import urllib.request
from collections import Counter, OrderedDict, defaultdict, deque, namedtuple
from contextlib import contextmanager
from datetime import UTC, date, datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

print(f"Python {sys.version.split()[0]}")
assert sys.version_info >= (3, 12), "This notebook needs Python 3.12+ (itertools.batched, datetime.UTC)."

random.seed(42)
OUTPUT_DIR = Path("_outputs")              # files we write go here (ignored by git)
OUTPUT_DIR.mkdir(exist_ok=True)


def check(name, got, expected, hint=""):
    """✅ if correct, ⏳ if not attempted yet (None), ❌ AssertionError with a hint otherwise."""
    if got is None or got is ...:
        print(f"⏳ {name}: not attempted yet — replace None with your answer.")
        return
    if isinstance(expected, float) and isinstance(got, (int, float)):
        ok = math.isclose(got, expected, rel_tol=1e-6, abs_tol=1e-9)
    else:
        ok = got == expected
    assert ok, f"❌ {name}: got {got!r} — not quite. {hint}"
    print(f"✅ {name}: correct!")

Python 3.12.11


## 1. Built-ins You Use Every Day 🟢

| Built-in | What it does | Tip |
|---|---|---|
| `len`, `sum`, `min`, `max` | size, total, smallest, largest | `max(..., key=...)`, `max([], default=None)` |
| `sorted(x, key=..., reverse=...)` | new sorted list | **stable**; sort by several fields with a tuple key |
| `any`, `all` | is at least one / every item truthy? | stop early; `all([])` is `True` |
| `enumerate`, `zip`, `reversed`, `range` | loop helpers | all **lazy** — they don't build lists |
| `isinstance`, `type` | type checks | `isinstance(x, (int, float))` |

A **key function** tells `sorted`/`min`/`max` *what to compare*. `lambda s: s[1]` is a tiny unnamed function that returns item 1 — section 2 explains `lambda`.

In [2]:
scores = [88, 92, 79, 92, 65]
print(len(scores), sum(scores), sum(scores, start=100), min(scores), max(scores))
print("max of an empty list with a default:", max([], default=None))

students = [("Ana", 92), ("Ben", 79), ("Chen", 92), ("Dev", 65)]
print("top student      :", max(students, key=lambda s: s[1]))          # ties → the FIRST one wins
print("score ↓ then name:", sorted(students, key=lambda s: (-s[1], s[0])))
print("anyone below 70? :", any(score < 70 for _, score in students))
print("everyone ≥ 60?   :", all(score >= 60 for _, score in students))
print("all([]) =", all([]), "| any([]) =", any([]))                   # "every item of nothing" is vacuously true

huge = range(0, 10**12, 2)                                             # half a trillion numbers...
print(f"range: len={len(huge):,}, 10**11 in it: {10**11 in huge}, size={sys.getsizeof(huge)} bytes")  # ...in 48 bytes
print(list(enumerate("abc", start=1)), list(zip("abc", [1, 2, 3])), list(reversed([1, 2, 3])))

5 416 516 65 92
max of an empty list with a default: None
top student      : ('Ana', 92)
score ↓ then name: [('Ana', 92), ('Chen', 92), ('Ben', 79), ('Dev', 65)]
anyone below 70? : True
everyone ≥ 60?   : True
all([]) = True | any([]) = False
range: len=500,000,000,000, 10**11 in it: True, size=48 bytes
[(1, 'a'), (2, 'b'), (3, 'c')] [('a', 1), ('b', 2), ('c', 3)] [3, 2, 1]


### ✍️ Your Turn

Return the file **names** sorted by size **largest first**; files with the same size should be in **alphabetical** order.

In [3]:
files = [("weights.pt", 540), ("data.csv", 120), ("notes.txt", 2), ("model.pt", 540)]
names_by_size = None  # TODO: your code here
check("names_by_size", names_by_size, ["model.pt", "weights.pt", "data.csv", "notes.txt"],
      hint="sorted(files, key=lambda f: (-f[1], f[0])), then keep only the names.")

⏳ names_by_size: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
files = [("weights.pt", 540), ("data.csv", 120), ("notes.txt", 2), ("model.pt", 540)]
names_by_size = [name for name, size in sorted(files, key=lambda f: (-f[1], f[0]))]
check("names_by_size", names_by_size, ["model.pt", "weights.pt", "data.csv", "notes.txt"])
```

Negating a number flips its order. For strings (which can't be negated) use **two stable sorts**: first by the secondary key, then by the primary key with `reverse=True`.
</details>

> 💡 **Interview angle:** "How does Python sort, and is it stable?" — `sorted`/`list.sort` use an adaptive merge sort (Timsort family): O(n log n) worst case, close to O(n) on nearly-sorted data, and **stable**, so equal keys keep their original order. The `key` function is called once per element.

## 2. Lambda, map, and filter 🟢

- **`lambda args: expression`** makes a small, unnamed function in one line. Great as a `key=` argument; if you'd give it a name, use `def` instead.
- **`map(func, items)`** applies `func` to every item. **`filter(func, items)`** keeps items where `func` returns truthy.
- Both return **lazy iterators** — nothing is computed until you loop over them. A comprehension usually reads better; `map` shines with an existing function like `map(int, parts)`.

In [4]:
square = lambda x: x * x                     # works, but style guides prefer: def square(x): return x * x
print(square(4))

parts = "3 14 15 92 65".split()
numbers_iter = map(int, parts)
print(numbers_iter)                          # a lazy map object — no ints exist yet
numbers = list(numbers_iter)                 # now they're computed
print(numbers)

print("filter       :", list(filter(lambda n: n % 2 == 0, numbers)))
print("comprehension:", [n for n in numbers if n % 2 == 0])       # same result, often clearer

fruits = ["banana", "apple", "Cherry"]
print(sorted(fruits), "vs", sorted(fruits, key=str.lower))       # by default ALL uppercase letters sort before lowercase

16
[3, 14, 15, 92, 65]
filter       : [14, 92]
comprehension: [14, 92]
['Cherry', 'apple', 'banana'] vs ['apple', 'banana', 'Cherry']


### ✍️ Your Turn

The sensor sends one line of comma-separated readings. Use **`map`** to turn them into floats and store their sum in `reading_total`.

In [5]:
line = "1.5, 2.25, 3"
reading_total = None  # TODO: your code here
check("reading_total", reading_total, 6.75, hint="sum(map(float, line.split(',')))  — float() ignores surrounding spaces.")

⏳ reading_total: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
line = "1.5, 2.25, 3"
reading_total = sum(map(float, line.split(",")))
check("reading_total", reading_total, 6.75)
```
</details>

> 💡 **Interview angle:** "`map`/`filter` or a comprehension?" — equivalent results; comprehensions are more readable when you'd need a `lambda`, and `map` is neat with an existing function. In Python 3 both `map` and `filter` are **lazy** iterators (in Python 2 they returned lists).

## 3. Iterators and the Iteration Protocol 🟡

- An **iterable** is anything you can loop over (list, string, dict, file, range). Calling `iter(iterable)` gives an **iterator**.
- An **iterator** remembers its position and hands out the next item with `next()`. When it runs out it raises `StopIteration`. It is **one-shot**: once exhausted, it stays empty.

A `for` loop is just this, hidden:

```
  for x in items:          iterator = iter(items)
      body(x)       ==     while True:
                               try:    x = next(iterator)
                               except StopIteration: break
                               body(x)
```

In [6]:
colors = ["red", "green", "blue"]
it = iter(colors)
print(next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("StopIteration: the iterator is exhausted")

# the for-loop machinery, written by hand
it = iter(colors)
while True:
    try:
        color = next(it)
    except StopIteration:
        break
    print("manual loop:", color)

# A list is iterable (fresh iterator every time) but NOT an iterator; a zip object IS an iterator
pairs = zip("ab", [1, 2])
print("first pass :", list(pairs))
print("second pass:", list(pairs), "← one-shot!")
print("list is its own iterator?", iter(colors) is colors, "| zip is?", iter(pairs) is pairs)

red green blue
StopIteration: the iterator is exhausted
manual loop: red
manual loop: green
manual loop: blue
first pass : [('a', 1), ('b', 2)]
second pass: [] ← one-shot!
list is its own iterator? False | zip is? True


> 💡 **Interview angle:** "Iterable vs iterator?" — an iterable has `__iter__` and can produce many independent iterators (a list); an iterator has `__next__` too, keeps position, and is consumed once (a generator, a file, `zip`, `map`).

## 4. Generators and Lazy Pipelines 🟡

A **generator function** contains `yield`. Calling it doesn't run the body — it returns a generator. Each `next()` runs the body until the next `yield`, hands out that value, and **pauses** (keeping all local variables) until asked again.

Think of a **tap vs a bucket**: a list is a bucket filled up front; a generator is a tap that gives water only while you hold your glass under it.

In [7]:
def countdown(n):
    print("  (body starts)")
    while n > 0:
        yield n                      # hand out n, pause here
        n -= 1                       # resume here on the next next()
    print("  (body ends → StopIteration)")


gen = countdown(3)
print("created:", gen)               # nothing printed from the body yet
print("next →", next(gen))
print("next →", next(gen))
print("rest →", list(gen))

created: <generator object countdown at 0x10b4c4940>
  (body starts)
next → 3
next → 2
  (body ends → StopIteration)
rest → [1]


**Measuring the memory saving.** `tracemalloc` (standard library) records the peak memory Python allocated while code ran.

In [8]:
def peak_memory_mb(func):
    tracemalloc.start()
    func()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1e6


N = 1_000_000
list_mb = peak_memory_mb(lambda: sum([x * x for x in range(N)]))    # builds a list of 1M ints first
gen_mb = peak_memory_mb(lambda: sum(x * x for x in range(N)))       # one value alive at a time
print(f"sum over a list      : peak {list_mb * 1000:10,.1f} KB")
print(f"sum over a generator : peak {gen_mb * 1000:10,.1f} KB → about {list_mb / max(gen_mb, 1e-9):,.0f}× less memory")

sum over a list      : peak   40,449.1 KB
sum over a generator : peak        0.5 KB → about 87,175× less memory


**Lazy pipelines.** Chain generators like an assembly line: each stage pulls one item from the previous stage only when it needs it. Here we count how much work is actually done when we only want the first 3 results.

In [9]:
work_done = Counter()


def read_records(n):
    for i in range(n):
        work_done["read"] += 1
        yield {"id": i, "value": (i * 37) % 101}      # a controlled toy data source for the demo


def only_large(records, threshold):
    for r in records:
        work_done["filtered"] += 1
        if r["value"] > threshold:
            yield r


def to_labels(records):
    for r in records:
        work_done["labelled"] += 1
        yield f"record {r['id']} (value {r['value']})"


pipeline = to_labels(only_large(read_records(1_000_000), threshold=90))
first_three = list(itertools.islice(pipeline, 3))      # take just 3 items from the (lazy) end of the pipeline
print(first_three)
print("work done:", dict(work_done), "← out of 1,000,000 records available")


def natural_numbers():                                  # an INFINITE generator is fine — just never list() it
    n = 1
    while True:
        yield n
        n += 1


def evens_then_odds(limit):
    yield from range(0, limit, 2)                       # yield from: delegate to another iterable
    yield from range(1, limit, 2)


print(list(itertools.islice(natural_numbers(), 5)), list(evens_then_odds(6)))

['record 8 (value 94)', 'record 19 (value 97)', 'record 30 (value 100)']
work done: {'read': 31, 'filtered': 31, 'labelled': 3} ← out of 1,000,000 records available
[1, 2, 3, 4, 5] [0, 2, 4, 1, 3, 5]


### ✍️ Your Turn

Write a generator `chunked(items, size)` that yields **lists** of `size` items; the last chunk may be shorter. (This is how you split a dataset into mini-batches.)

In [10]:
def chunked(items, size):
    # TODO: replace the next line with a generator that uses `yield`
    return None


chunks = chunked(range(7), 3)
check("chunked", None if chunks is None else [list(c) for c in chunks], [[0, 1, 2], [3, 4, 5], [6]],
      hint="Keep a `batch` list; append each item; when len(batch) == size, yield it and start a new one. Yield leftovers at the end.")

⏳ chunked: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def chunked(items, size):
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:                      # don't forget the final, shorter chunk
        yield batch


chunks = chunked(range(7), 3)
check("chunked", [list(c) for c in chunks], [[0, 1, 2], [3, 4, 5], [6]])
```

Python 3.12 ships this as `itertools.batched(items, 3)` (yielding tuples) — see section 10.
</details>

> 💡 **Interview angle:** "How would you process a 50 GB file on a laptop with 8 GB of RAM?" — iterate the file line by line (a file object is a lazy iterator), chain generator stages for parsing/filtering, and aggregate into small structures like a `Counter`. Memory stays roughly constant regardless of file size.

## 5. functools: partial, reduce, lru_cache 🟡

`functools` holds tools that take or return functions ("higher-order functions"):

| Tool | What it does |
|---|---|
| `partial(func, *args, **kwargs)` | a new function with some arguments pre-filled |
| `reduce(func, items, initial)` | fold a sequence into one value, left to right |
| `lru_cache(maxsize=128)` / `cache` | remember results of previous calls (**memoization**) |
| `wraps(func)` | copy name/docstring onto a wrapper (section 7) |

In [11]:
parse_binary = functools.partial(int, base=2)          # int(text, base=2) with base pre-filled
round_money = functools.partial(round, ndigits=2)
print(parse_binary("1011"), round_money(3.14159))

product = functools.reduce(lambda acc, x: acc * x, [1, 2, 3, 4, 5], 1)
print("reduce product:", product, "| math.prod:", math.prod([1, 2, 3, 4, 5]), "← prefer the built-in when one exists")

nested_config = {"model": {"encoder": {"layers": 12}}}
print("deep lookup with reduce:", functools.reduce(lambda d, key: d[key], ["model", "encoder", "layers"], nested_config))

11 3.14
reduce product: 120 | math.prod: 120 ← prefer the built-in when one exists
deep lookup with reduce: 12


**Memoization with `lru_cache`.** The naive Fibonacci function recomputes the same values exponentially many times. Caching makes it linear. *LRU* = "least recently used": when the cache is full, the entry unused for the longest time is thrown out.

In [12]:
def fib_plain(n):
    return n if n < 2 else fib_plain(n - 1) + fib_plain(n - 2)


@functools.lru_cache(maxsize=None)                     # maxsize=None → unlimited (same as @functools.cache)
def fib_cached(n):
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)


start = time.perf_counter(); plain_result = fib_plain(30); plain_s = time.perf_counter() - start
start = time.perf_counter(); cached_result = fib_cached(30); cached_s = time.perf_counter() - start
assert plain_result == cached_result
print(f"fib(30) = {plain_result:,} | plain {plain_s * 1000:.0f} ms | cached {cached_s * 1000:.3f} ms "
      f"→ {plain_s / cached_s:,.0f}× faster")
print(fib_cached.cache_info())
print("fib(300) with the cache is instant:", str(fib_cached(300))[:20] + "...")

try:
    fib_cached([30])                                   # arguments become dict keys → must be hashable
except TypeError as err:
    print("TypeError:", err)

fib(30) = 832,040 | plain 47 ms | cached 0.016 ms → 2,950× faster
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)
fib(300) with the cache is instant: 22223224462942044552...
TypeError: unhashable type: 'list'


### ✍️ Your Turn

Use `functools.partial` to create `parse_hex`, a function that turns hexadecimal text like `"ff"` into an int.

In [13]:
parse_hex = None  # TODO: your code here
check("parse_hex", None if parse_hex is None else [parse_hex("ff"), parse_hex("10")], [255, 16],
      hint="int accepts a `base` keyword argument.")

⏳ parse_hex: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
parse_hex = functools.partial(int, base=16)
check("parse_hex", [parse_hex("ff"), parse_hex("10")], [255, 16])
```
</details>

> 💡 **Interview angle:** "When should you *not* use `lru_cache`?" — when arguments aren't hashable, when the function has side effects or depends on changing outside state (you'd get stale results), when results are huge (unbounded memory), or on instance methods where the cache keeps `self` alive.

## 6. Closures 🟡

Functions can be defined **inside** other functions. A **closure** is an inner function that **remembers variables from the enclosing function**, even after that outer function has returned — like a backpack of variables the inner function carries around.

```
  make_multiplier(3) ──returns──►  multiply(x)  +  backpack {factor: 3}
```

In [14]:
def make_multiplier(factor):
    def multiply(x):
        return x * factor            # `factor` comes from the enclosing scope
    return multiply


triple = make_multiplier(3)
halve = make_multiplier(0.5)
print(triple(10), halve(10))
print("triple's backpack:", [cell.cell_contents for cell in triple.__closure__])


def make_counter():
    count = 0

    def increment():
        nonlocal count               # assign to the ENCLOSING variable, not a new local one
        count += 1
        return count
    return increment


counter_a, counter_b = make_counter(), make_counter()
print("a:", counter_a(), counter_a(), counter_a(), "| b:", counter_b(), "← each closure has its own state")

30 5.0
triple's backpack: [3]
a: 1 2 3 | b: 1 ← each closure has its own state


**Late binding.** A closure remembers the *variable*, not the value it had when the function was created — a famous interview trap.

In [15]:
callbacks = [lambda: i for i in range(3)]             # every lambda looks up `i` when CALLED
print("late binding  :", [f() for f in callbacks])     # the loop is over → i == 2 for all

callbacks = [lambda i=i: i for i in range(3)]         # default values are evaluated NOW → captures each value
print("captured value:", [f() for f in callbacks])

callbacks = [functools.partial(lambda i: i, i) for i in range(3)]   # partial also freezes the value
print("with partial  :", [f() for f in callbacks])

late binding  : [2, 2, 2]
captured value: [0, 1, 2]
with partial  : [0, 1, 2]


### ✍️ Your Turn

Write `make_accumulator()` that returns a function `add(amount)`; each call adds `amount` to a running total and **returns the new total**. Two accumulators must not share their totals.

In [16]:
def make_accumulator():
    # TODO: replace the next line — define an inner `add(amount)` using nonlocal, and return it
    return None


acc1, acc2 = make_accumulator(), make_accumulator()
check("accumulator", None if acc1 is None else [acc1(10), acc1(5), acc2(1), acc1(-3)], [10, 15, 1, 12],
      hint="total = 0; def add(amount): nonlocal total; total += amount; return total")

⏳ accumulator: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def make_accumulator():
    total = 0

    def add(amount):
        nonlocal total
        total += amount
        return total
    return add


acc1, acc2 = make_accumulator(), make_accumulator()
check("accumulator", [acc1(10), acc1(5), acc2(1), acc1(-3)], [10, 15, 1, 12])
```
</details>

> 💡 **Interview angle:** "What does `[lambda: i for i in range(3)]` return when called?" — `[2, 2, 2]`: closures look variables up at call time. Fix with a default argument `lambda i=i: i` or `functools.partial`.

## 7. Decorators 🔴

A **decorator** is a function that **takes a function and returns a new function** (usually a *wrapper* that runs extra code before/after the original). The `@` line is just shorthand:

```
  @timer                            def slow_sum(n): ...
  def slow_sum(n):          ==      slow_sum = timer(slow_sum)
      ...

  call slow_sum(10)  →  wrapper(10)  →  [start clock] → original slow_sum(10) → [stop clock] → return result
```

Decorators rely on everything so far: functions as values, `*args/**kwargs`, and closures.

In [17]:
def timer(func):
    @functools.wraps(func)                             # copy func's name, docstring, etc. onto wrapper
    def wrapper(*args, **kwargs):                      # accept ANY arguments and pass them through
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed_ms = (time.perf_counter() - start) * 1000
        print(f"  ⏱️ {func.__name__}{args} took {elapsed_ms:.2f} ms")
        return result                                  # don't forget to return the original result!
    return wrapper


@timer
def slow_sum(n):
    """Add the numbers 0..n-1 with a Python loop."""
    total = 0
    for i in range(n):
        total += i
    return total


print("result:", slow_sum(1_000_000))
print("name:", slow_sum.__name__, "| doc:", slow_sum.__doc__, "| original still reachable:", slow_sum.__wrapped__.__name__)

  ⏱️ slow_sum(1000000,) took 15.45 ms
result: 499999500000
name: slow_sum | doc: Add the numbers 0..n-1 with a Python loop. | original still reachable: slow_sum


In [18]:
def timer_without_wraps(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@timer_without_wraps
def predict(features):
    """Return a prediction for one row."""
    return sum(features)


print("without wraps → name:", predict.__name__, "| doc:", predict.__doc__, "← logs, help(), and debuggers now see 'wrapper'")

without wraps → name: wrapper | doc: None ← logs, help(), and debuggers now see 'wrapper'


**Decorators with arguments** need **one more layer**: the outer function receives the settings and returns the actual decorator.

```
  @retry(times=3)   →   retry(times=3) returns `decorator`   →   decorator(func) returns `wrapper`
```

`retry` is a real-world favourite: network calls to APIs (LLM providers, databases) fail transiently.

In [19]:
def retry(times=3, exceptions=(Exception,), delay_s=0.0):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as err:
                    print(f"  attempt {attempt}/{times} failed: {type(err).__name__}: {err}")
                    if attempt == times:
                        raise                          # out of attempts: re-raise the last error
                    time.sleep(delay_s * 2 ** (attempt - 1))   # exponential backoff
        return wrapper
    return decorator


calls = {"n": 0}


@retry(times=3, exceptions=(ConnectionError,), delay_s=0.01)
def flaky_api_call():
    """Fails on the first two calls (a controlled stand-in for a flaky network), then succeeds."""
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("temporary network glitch")
    return {"status": "ok", "attempts": calls["n"]}


print("returned:", flaky_api_call())


@retry(times=2, exceptions=(ConnectionError,))
def always_down():
    raise ConnectionError("service unavailable")


try:
    always_down()
except ConnectionError as err:
    print("gave up → ConnectionError:", err)

  attempt 1/3 failed: ConnectionError: temporary network glitch
  attempt 2/3 failed: ConnectionError: temporary network glitch
returned: {'status': 'ok', 'attempts': 3}
  attempt 1/2 failed: ConnectionError: service unavailable
  attempt 2/2 failed: ConnectionError: service unavailable
gave up → ConnectionError: service unavailable


In [20]:
def bold(func):
    @functools.wraps(func)
    def wrapper():
        return f"<b>{func()}</b>"
    return wrapper


def italic(func):
    @functools.wraps(func)
    def wrapper():
        return f"<i>{func()}</i>"
    return wrapper


@bold                                                  # applied SECOND (outermost)
@italic                                                # applied FIRST (closest to the function)
def greeting():
    return "hello"


print("stacked:", greeting(), "== bold(italic(greeting))")

stacked: <b><i>hello</i></b> == bold(italic(greeting))


### ✍️ Your Turn

Complete the `count_calls` decorator: the wrapper should count how many times the function was called in an attribute `wrapper.calls`, and keep the original name and docstring.

In [21]:
def count_calls(func):
    # TODO: define a wrapper (with @functools.wraps(func)) that increments wrapper.calls,
    #       set wrapper.calls = 0, and return the wrapper instead of func
    return func


@count_calls
def add(a, b):
    """Add two numbers."""
    return a + b


for pair in [(1, 2), (3, 4), (5, 6)]:
    add(*pair)
n_calls = getattr(add, "calls", None)
check("count_calls", None if n_calls is None else (n_calls, add.__name__, add.__doc__), (3, "add", "Add two numbers."),
      hint="Functions are objects, so you can set wrapper.calls = 0 and do wrapper.calls += 1 inside the wrapper.")

⏳ count_calls: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper


@count_calls
def add(a, b):
    """Add two numbers."""
    return a + b


for pair in [(1, 2), (3, 4), (5, 6)]:
    add(*pair)
check("count_calls", (add.calls, add.__name__, add.__doc__), (3, "add", "Add two numbers."))
```
</details>

> 💡 **Interview angle:** "Write a decorator that takes arguments" — say the three layers out loud: *factory(settings) → decorator(func) → wrapper(\*args, \*\*kwargs)*, use `functools.wraps`, and return the wrapped function's result.

## 8. Context Managers 🟡

A **context manager** guarantees that cleanup happens after a block of code — **even if the block raises an exception**. The `with` statement calls two methods:

```
  with manager as value:        value = manager.__enter__()      ← setup
      body                      body
                                manager.__exit__(exc_type, exc, traceback)   ← ALWAYS runs; returning True swallows the error
```

There are two ways to write one: a **class** with `__enter__`/`__exit__`, or a **generator** decorated with `@contextmanager`.

In [22]:
class ChangeDirectory:
    """Temporarily switch the working directory; always switch back."""

    def __init__(self, target):
        self.target = Path(target)

    def __enter__(self):
        self.previous = Path.cwd()
        os.chdir(self.target)
        return self.target

    def __exit__(self, exc_type, exc, traceback):
        os.chdir(self.previous)                        # runs on success AND on error
        return False                                   # False → don't swallow exceptions


start_dir = Path.cwd()
with ChangeDirectory(OUTPUT_DIR):
    print("inside :", Path.cwd().name)
print("after  :", Path.cwd() == start_dir)

try:
    with ChangeDirectory(OUTPUT_DIR):
        raise RuntimeError("something failed inside the block")
except RuntimeError as err:
    print("error propagated:", err, "| directory restored anyway:", Path.cwd() == start_dir)

with contextlib.chdir(OUTPUT_DIR):                     # the standard library has this exact tool (3.11+)
    print("contextlib.chdir inside:", Path.cwd().name)
print("contextlib.chdir restored:", Path.cwd() == start_dir)

inside : _outputs
after  : True
error propagated: something failed inside the block | directory restored anyway: True
contextlib.chdir inside: _outputs
contextlib.chdir restored: True


**`@contextmanager`**: code before `yield` is the setup, code after it is the cleanup. The cleanup **must be in a `finally:` block** — otherwise an exception inside the `with` block is raised *at the `yield`* and the cleanup line never runs.

In [23]:
@contextmanager
def temporary_setting_buggy(settings, key, value):
    old = settings[key]
    settings[key] = value
    yield                                              # ❌ an exception in the block is raised HERE...
    settings[key] = old                                # ...so this restore line is skipped


@contextmanager
def temporary_setting(settings, key, value):
    old = settings[key]
    settings[key] = value
    try:
        yield settings
    finally:
        settings[key] = old                            # ✅ always restored


for manager in (temporary_setting_buggy, temporary_setting):
    config = {"dropout": 0.1}
    try:
        with manager(config, "dropout", 0.0):
            raise ValueError("evaluation crashed")
    except ValueError:
        pass
    print(f"{manager.__name__:>24}: dropout after the crash = {config['dropout']}")

 temporary_setting_buggy: dropout after the crash = 0.0
       temporary_setting: dropout after the crash = 0.1


In [24]:
with contextlib.suppress(FileNotFoundError):           # ignore one specific, expected error
    Path("definitely_missing.txt").unlink()
print("suppress: missing file ignored")

with open(OUTPUT_DIR / "a.txt", "w", encoding="utf-8") as fa, open(OUTPUT_DIR / "b.txt", "w", encoding="utf-8") as fb:
    fa.write("A"); fb.write("B")                        # several managers in one `with`
print("both files closed:", fa.closed and fb.closed)

suppress: missing file ignored
both files closed: True


### ✍️ Your Turn

Write a `@contextmanager` called `logged_block(log)` that appends `"start"` to `log` before the block and `"end"` after it — **even when the block raises**.

In [25]:
@contextmanager
def logged_block(log):
    # TODO: replace the next line — append "start", then yield inside try, append "end" in finally
    raise NotImplementedError
    yield


events_log = []
try:
    with logged_block(events_log):
        events_log.append("work")
        raise ValueError("boom")
except ValueError:
    pass
except NotImplementedError:
    events_log = None
check("logged_block", events_log, ["start", "work", "end"], hint="Put `yield` inside try: and the 'end' append in finally:")

⏳ logged_block: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
@contextmanager
def logged_block(log):
    log.append("start")
    try:
        yield
    finally:
        log.append("end")


events_log = []
try:
    with logged_block(events_log):
        events_log.append("work")
        raise ValueError("boom")
except ValueError:
    pass
check("logged_block", events_log, ["start", "work", "end"])
```
</details>

> 💡 **Interview angle:** "How does `with` work?" — it calls `__enter__`, binds its return value to `as`, runs the block, then always calls `__exit__` with the exception info (or `None`s). A truthy return from `__exit__` suppresses the exception. With `@contextmanager`, cleanup belongs in `finally`.

## 9. collections: Counter, defaultdict, deque, namedtuple 🟢

| Class | Use it when you need… | Replaces |
|---|---|---|
| `Counter` | counts of things, top-k | `d[k] = d.get(k, 0) + 1` loops |
| `defaultdict(list)` | grouping without checking "is the key there yet?" | `setdefault` / `if k not in d` |
| `deque` | fast add/remove at **both ends**, fixed-size windows | `list.pop(0)` (O(n)) |
| `namedtuple` | a small immutable record with named fields | plain tuples with magic indexes |

In [26]:
labels = ["cat", "dog", "cat", "bird", "cat", "dog"]
label_counts = Counter(labels)
print(label_counts, "| top 2:", label_counts.most_common(2), "| total:", label_counts.total())
label_counts.update(["bird", "bird"])                  # add more observations
print("after update:", label_counts, "| missing key → 0:", label_counts["fish"])
print("counter math:", Counter(a=3, b=1) - Counter(a=1, b=2), "← counts ≤ 0 are dropped")

by_first_letter = defaultdict(list)                     # missing keys start as list()
for word in ["apple", "avocado", "banana", "blueberry", "cherry"]:
    by_first_letter[word[0]].append(word)
print(dict(by_first_letter))

Counter({'cat': 3, 'dog': 2, 'bird': 1}) | top 2: [('cat', 3), ('dog', 2)] | total: 6
after update: Counter({'cat': 3, 'bird': 3, 'dog': 2}) | missing key → 0: 0
counter math: Counter({'a': 2}) ← counts ≤ 0 are dropped
{'a': ['apple', 'avocado'], 'b': ['banana', 'blueberry'], 'c': ['cherry']}


In [27]:
recent = deque(maxlen=3)                               # keeps only the last 3 items — a sliding window
for reading in [10, 20, 30, 40, 50]:
    recent.append(reading)
    print(f"  added {reading} → window {list(recent)} → mean {sum(recent) / len(recent):.1f}")

queue = deque(["job1", "job2"])
queue.appendleft("urgent")
queue.append("job3")
print("queue:", list(queue), "| served:", queue.popleft())

n = 50_000
as_list, as_deque = list(range(n)), deque(range(n))
start = time.perf_counter()
while as_list:
    as_list.pop(0)                                     # O(n) each: shifts every remaining item
list_s = time.perf_counter() - start
start = time.perf_counter()
while as_deque:
    as_deque.popleft()                                 # O(1) each
deque_s = time.perf_counter() - start
print(f"emptying {n:,} items from the front: list.pop(0) {list_s * 1000:.0f} ms | deque.popleft {deque_s * 1000:.1f} ms "
      f"→ {list_s / deque_s:.0f}× faster")

Prediction = namedtuple("Prediction", ["label", "score"])
pred = Prediction("spam", 0.97)
print(pred, "| pred.label:", pred.label, "| unpacks:", tuple(pred), "| as dict:", pred._asdict())
print("changed copy:", pred._replace(score=0.5), "| original untouched:", pred)

  added 10 → window [10] → mean 10.0
  added 20 → window [10, 20] → mean 15.0
  added 30 → window [10, 20, 30] → mean 20.0
  added 40 → window [20, 30, 40] → mean 30.0
  added 50 → window [30, 40, 50] → mean 40.0
queue: ['urgent', 'job1', 'job2', 'job3'] | served: urgent


emptying 50,000 items from the front: list.pop(0) 137 ms | deque.popleft 1.4 ms → 97× faster
Prediction(label='spam', score=0.97) | pred.label: spam | unpacks: ('spam', 0.97) | as dict: {'label': 'spam', 'score': 0.97}
changed copy: Prediction(label='spam', score=0.5) | original untouched: Prediction(label='spam', score=0.97)


### ✍️ Your Turn

Use a `defaultdict` to group model names by their framework. Store a normal `dict` in `models_by_framework` (convert with `dict(...)`).

In [28]:
models = [("resnet", "pytorch"), ("bert", "pytorch"), ("xgb-baseline", "xgboost"), ("vit", "pytorch"), ("lgbm", "lightgbm")]
models_by_framework = None  # TODO: your code here
check("models_by_framework", models_by_framework,
      {"pytorch": ["resnet", "bert", "vit"], "xgboost": ["xgb-baseline"], "lightgbm": ["lgbm"]},
      hint="groups = defaultdict(list); for name, fw in models: groups[fw].append(name)")

⏳ models_by_framework: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
models = [("resnet", "pytorch"), ("bert", "pytorch"), ("xgb-baseline", "xgboost"), ("vit", "pytorch"), ("lgbm", "lightgbm")]
groups = defaultdict(list)
for name, framework in models:
    groups[framework].append(name)
models_by_framework = dict(groups)
check("models_by_framework", models_by_framework,
      {"pytorch": ["resnet", "bert", "vit"], "xgboost": ["xgb-baseline"], "lightgbm": ["lgbm"]})
```
</details>

> 💡 **Interview angle:** "Implement a queue / BFS in Python" — use `collections.deque` with `append` + `popleft` (both O(1)). A list with `pop(0)` is O(n) per pop, as measured above.

## 10. itertools 🟡

`itertools` provides fast, memory-efficient **iterator building blocks**. They're lazy, so wrap them in `list()` to see the values.

| Function | Produces | Example |
|---|---|---|
| `chain(a, b)` | a then b | `chain([1, 2], [3])` → 1 2 3 |
| `islice(it, stop)` | a slice of any iterator | first n items of a generator |
| `groupby(it, key)` | runs of **consecutive** equal keys | ⚠️ sort by the key first! |
| `product`, `combinations`, `permutations` | combinatorics | hyper-parameter grids, pairs |
| `accumulate`, `pairwise`, `batched` | running totals, neighbours, fixed-size chunks | cumulative sums, deltas, mini-batches |

In [29]:
print("chain       :", list(itertools.chain([1, 2], (3, 4), "ab")))
print("islice      :", list(itertools.islice(itertools.count(start=10, step=5), 4)))
print("accumulate  :", list(itertools.accumulate([3, 1, 4, 1, 5])))
print("pairwise    :", list(itertools.pairwise([10, 13, 19, 20])))
print("batched     :", list(itertools.batched(range(7), 3)))

grid = {"lr": [0.1, 0.01], "batch_size": [32, 64]}
print("\nhyper-parameter grid (product):")
for lr, batch_size in itertools.product(grid["lr"], grid["batch_size"]):
    print(f"  lr={lr}, batch_size={batch_size}")
print("combinations(3 models, 2):", list(itertools.combinations(["a", "b", "c"], 2)))
print("permutations('abc', 2)   :", len(list(itertools.permutations("abc", 2))), "ordered pairs")

chain       : [1, 2, 3, 4, 'a', 'b']
islice      : [10, 15, 20, 25]
accumulate  : [3, 4, 8, 9, 14]
pairwise    : [(10, 13), (13, 19), (19, 20)]
batched     : [(0, 1, 2), (3, 4, 5), (6,)]

hyper-parameter grid (product):
  lr=0.1, batch_size=32
  lr=0.1, batch_size=64
  lr=0.01, batch_size=32
  lr=0.01, batch_size=64
combinations(3 models, 2): [('a', 'b'), ('a', 'c'), ('b', 'c')]
permutations('abc', 2)   : 6 ordered pairs


**The `groupby` caveat:** it only groups items that are **next to each other**. Unsorted input splits a group into several pieces.

In [30]:
purchases = [("fruit", "apple"), ("veg", "kale"), ("fruit", "pear"), ("veg", "leek"), ("fruit", "fig")]
category = lambda p: p[0]

print("❌ unsorted:", [(key, [item for _, item in group]) for key, group in itertools.groupby(purchases, key=category)])
print("✅ sorted  :", [(key, [item for _, item in group]) for key, group in itertools.groupby(sorted(purchases, key=category), key=category)])

❌ unsorted: [('fruit', ['apple']), ('veg', ['kale']), ('fruit', ['pear']), ('veg', ['leek']), ('fruit', ['fig'])]
✅ sorted  : [('fruit', ['apple', 'pear', 'fig']), ('veg', ['kale', 'leek'])]


### ✍️ Your Turn

Split `range(10)` into mini-batches of 4 with `itertools.batched` and store **each batch's sum** in `batch_sums`.

In [31]:
batch_sums = None  # TODO: your code here
check("batch_sums", batch_sums, [6, 22, 17], hint="[sum(batch) for batch in itertools.batched(range(10), 4)]")

⏳ batch_sums: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
batch_sums = [sum(batch) for batch in itertools.batched(range(10), 4)]
check("batch_sums", batch_sums, [6, 22, 17])
```
</details>

> 💡 **Interview angle:** "Your `groupby` report shows the same category several times — why?" — `itertools.groupby` groups *consecutive* items only (like Unix `uniq`); sort by the same key first, or use a `defaultdict(list)` when order doesn't matter.

## 11. Dates and Times: datetime and zoneinfo 🟡

- `date` (a day), `datetime` (a day + time), `timedelta` (a duration).
- **Naive** datetimes have no time zone; **aware** ones carry `tzinfo`. Rule of thumb: **store and compute in UTC, convert to local time only for display.**
- `zoneinfo.ZoneInfo("Asia/Kolkata")` gives real time-zone rules (including daylight saving time) from the IANA database.
- Parse with `datetime.fromisoformat` / `strptime`; format with `isoformat` / `strftime`.

In [32]:
launch = date(2024, 2, 28)
print("two days later:", launch + timedelta(days=2), "← 2024 is a leap year, so Feb 29 exists")
days_in_2024 = (date(2025, 1, 1) - date(2024, 1, 1)).days
print("days in 2024:", days_in_2024)                   # computed, not typed

stamp = datetime.strptime("Sun Dec 04 04:47:44 2005", "%a %b %d %H:%M:%S %Y")   # parse a custom format
print("parsed:", stamp, "| weekday:", stamp.strftime("%A"), "| ISO:", stamp.isoformat())
print("from ISO text:", datetime.fromisoformat("2026-09-13T18:30:00+05:30"))
print("difference:", datetime(2026, 1, 1, 9, 30) - datetime(2025, 12, 31, 22, 0))

two days later: 2024-03-01 ← 2024 is a leap year, so Feb 29 exists
days in 2024: 366
parsed: 2005-12-04 04:47:44 | weekday: Sunday | ISO: 2005-12-04T04:47:44
from ISO text: 2026-09-13 18:30:00+05:30
difference: 11:30:00


In [33]:
now_utc = datetime.now(UTC)                            # aware "now" (datetime.utcnow() is deprecated since 3.12)
print("now (UTC)        :", now_utc.isoformat(timespec="seconds"))
print("same moment, Delhi:", now_utc.astimezone(ZoneInfo("Asia/Kolkata")).isoformat(timespec="seconds"))
print("same moment, NYC  :", now_utc.astimezone(ZoneInfo("America/New_York")).isoformat(timespec="seconds"))

naive = datetime(2026, 1, 1, 12, 0)
try:
    naive < now_utc
except TypeError as err:
    print("TypeError:", err)

# Daylight saving time: in New York, clocks jump from 02:00 to 03:00 on 2026-03-08
new_york = ZoneInfo("America/New_York")
before = datetime(2026, 3, 7, 12, 0, tzinfo=new_york)
after = datetime(2026, 3, 8, 12, 0, tzinfo=new_york)
print("wall-clock difference :", after - before)                                   # same tzinfo → wall-clock math
print("real elapsed time     :", after.astimezone(UTC) - before.astimezone(UTC))   # UTC → true duration

now (UTC)        : 2026-09-13T22:40:38+00:00
same moment, Delhi: 2026-09-14T04:10:38+05:30
same moment, NYC  : 2026-09-13T18:40:38-04:00
TypeError: can't compare offset-naive and offset-aware datetimes
wall-clock difference : 1 day, 0:00:00
real elapsed time     : 23:00:00


### ✍️ Your Turn

How many **days** are there from `"2026-01-15"` to `"2026-03-01"`? Parse both strings and store the integer in `days_between`.

In [34]:
start_text, end_text = "2026-01-15", "2026-03-01"
days_between = None  # TODO: your code here
check("days_between", days_between, 45, hint="(date.fromisoformat(end_text) - date.fromisoformat(start_text)).days")

⏳ days_between: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
start_text, end_text = "2026-01-15", "2026-03-01"
days_between = (date.fromisoformat(end_text) - date.fromisoformat(start_text)).days
check("days_between", days_between, 45)
```
</details>

> 💡 **Interview angle:** "How should a service store timestamps?" — as timezone-aware UTC (e.g. ISO 8601 with `+00:00`, or epoch seconds), converting to the user's zone only at the edges. Naive local times break across DST changes and servers in different regions.

## 12. JSON 🟢

**JSON** (JavaScript Object Notation) is the text format of web APIs, config files, and LLM tool calls.

| Python | JSON | Round trip gotcha |
|---|---|---|
| `dict` | object `{}` | **keys become strings** |
| `list`, `tuple` | array `[]` | tuples come back as lists |
| `str`, `int`, `float` | string, number | — |
| `True` / `False` / `None` | `true` / `false` / `null` | — |
| `datetime`, `set`, custom objects | ❌ not supported | convert first (`isoformat()`, `list()`) |

In [35]:
run = {"model": "bert", "f1": 0.912, "tags": ("nlp", "baseline"), "epochs_done": {1: 0.8, 2: 0.9}, "notes": None}
text = json.dumps(run, indent=2)
print(text)
back = json.loads(text)
print("tuple → list:", back["tags"], "| int keys → str:", list(back["epochs_done"]))

try:
    json.dumps({"finished_at": datetime(2026, 9, 13, 18, 30, tzinfo=UTC)})
except TypeError as err:
    print("TypeError:", err)
print(json.dumps({"finished_at": datetime(2026, 9, 13, 18, 30, tzinfo=UTC).isoformat()}))   # convert explicitly

# JSON Lines: one JSON object per line — the usual format for datasets and LLM fine-tuning files
records = [{"prompt": "2+2?", "answer": "4"}, {"prompt": "Capital of France?", "answer": "Paris"}]
jsonl_path = OUTPUT_DIR / "qa.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record) + "\n")
with open(jsonl_path, encoding="utf-8") as f:
    loaded = [json.loads(line) for line in f]           # stream it back line by line
print("JSON Lines round trip OK:", loaded == records)

{
  "model": "bert",
  "f1": 0.912,
  "tags": [
    "nlp",
    "baseline"
  ],
  "epochs_done": {
    "1": 0.8,
    "2": 0.9
  },
  "notes": null
}
tuple → list: ['nlp', 'baseline'] | int keys → str: ['1', '2']
TypeError: Object of type datetime is not JSON serializable
{"finished_at": "2026-09-13T18:30:00+00:00"}
JSON Lines round trip OK: True


> 💡 **Interview angle:** "Why does my dict come back from JSON with string keys?" — JSON object keys are always strings. Convert keys back explicitly (`{int(k): v for k, v in d.items()}`) or store a list of records instead.

## 13. os, sys, and pathlib 🟢

- **`os`** talks to the operating system: environment variables, the current directory, CPU count.
- **`sys`** talks to the Python interpreter: version, which executable is running, where imports are searched (`sys.path`).
- **`pathlib`** is the modern way to handle paths (prefer it over the older `os.path` functions).

| Old `os.path` style | `pathlib` style |
|---|---|
| `os.path.join(a, "f.txt")` | `Path(a) / "f.txt"` |
| `os.path.exists(p)` | `p.exists()` |
| `os.path.splitext(p)[1]` | `p.suffix` |
| `os.makedirs(d, exist_ok=True)` | `d.mkdir(parents=True, exist_ok=True)` |
| `glob.glob("*.csv")` | `Path(".").glob("*.csv")` |

In [36]:
api_key = os.environ.get("OPENAI_API_KEY")             # secrets come from environment variables, never from code
print("OPENAI_API_KEY is set:", api_key is not None)   # print whether it exists — never the key itself
print("current folder name:", Path.cwd().name, "| CPUs:", os.cpu_count())

print("python version tuple:", sys.version_info[:3], "| platform:", sys.platform)
print("running interpreter:", Path(sys.executable).name, "| import search paths:", len(sys.path))

project = OUTPUT_DIR / "demo_project"
(project / "data" / "raw").mkdir(parents=True, exist_ok=True)
for name in ["train.csv", "test.csv", "README.md"]:
    (project / "data" / "raw" / name).write_text("placeholder\n", encoding="utf-8")

print("\nfiles (recursive):", sorted(str(p.relative_to(project)) for p in project.rglob("*") if p.is_file()))
print("csv files only   :", sorted(p.name for p in project.rglob("*.csv")))
report = project / "data" / "raw" / "train.csv"
print("stem / suffix / new suffix:", report.stem, "/", report.suffix, "/", report.with_suffix(".parquet").name)

OPENAI_API_KEY is set: False
current folder name: 00_Foundations | CPUs: 15
python version tuple: (3, 12, 11) | platform: darwin
running interpreter: python | import search paths: 5

files (recursive): ['data/raw/README.md', 'data/raw/test.csv', 'data/raw/train.csv']
csv files only   : ['test.csv', 'train.csv']
stem / suffix / new suffix: train / .csv / train.parquet


> 💡 **Interview angle:** "How do you handle API keys in a Python service?" — read them from environment variables (or a secrets manager) with `os.environ`, fail with a clear message when missing, never commit them, and never log them.

## 🔧 Build It From Scratch

Re-implementing built-ins is a classic interview exercise and the best way to *really* understand iterators, generators, decorators, and context managers. Each version below is `assert`-ed against the real thing.

### 1) `my_map` and `my_zip` — lazy, like the originals

In [37]:
def my_map(func, *iterables):
    iterators = [iter(it) for it in iterables]
    if not iterators:
        raise TypeError("my_map() must have at least two arguments.")
    while True:
        try:
            args = [next(it) for it in iterators]
        except StopIteration:                          # the shortest input ran out
            return
        yield func(*args)


def my_zip(*iterables):
    iterators = [iter(it) for it in iterables]
    if not iterators:                                  # zip() with no arguments yields nothing
        return
    while True:
        try:
            items = [next(it) for it in iterators]
        except StopIteration:
            return
        yield tuple(items)


for _ in range(100):
    a = [random.randint(0, 9) for _ in range(random.randint(0, 8))]
    b = [random.randint(0, 9) for _ in range(random.randint(0, 8))]
    assert list(my_map(abs, a)) == list(map(abs, a))
    assert list(my_map(pow, a, b)) == list(map(pow, a, b))
    assert list(my_zip(a, b, "xyz")) == list(zip(a, b, "xyz"))
assert list(my_zip()) == list(zip())
assert list(itertools.islice(my_zip(itertools.count(), "ab"), 5)) == [(0, "a"), (1, "b")]   # lazy: works on infinite input
print("✅ my_map and my_zip match map and zip (including uneven lengths and infinite iterators)")

✅ my_map and my_zip match map and zip (including uneven lengths and infinite iterators)


### 2) `my_range` as a generator

In [38]:
def my_range(start, stop=None, step=1):
    if stop is None:                                   # my_range(5) means my_range(0, 5)
        start, stop = 0, start
    if step == 0:
        raise ValueError("my_range() arg 3 must not be zero")
    current = start
    while (current < stop) if step > 0 else (current > stop):
        yield current
        current += step


for _ in range(300):
    start_, stop_ = random.randint(-10, 10), random.randint(-10, 10)
    step_ = random.choice([-3, -2, -1, 1, 2, 3])
    assert list(my_range(start_, stop_, step_)) == list(range(start_, stop_, step_))
    assert list(my_range(stop_)) == list(range(stop_))
for impl in (range, my_range):
    try:
        list(impl(0, 5, 0))
    except ValueError as err:
        print(f"{impl.__name__}(0, 5, 0) → ValueError: {err}")
print("✅ my_range matches range on 300 random (start, stop, step) combinations")

range(0, 5, 0) → ValueError: range() arg 3 must not be zero
my_range(0, 5, 0) → ValueError: my_range() arg 3 must not be zero
✅ my_range matches range on 300 random (start, stop, step) combinations


### 3) An `lru_cache`-style memoize decorator

An **LRU cache** keeps the most recently used results. An `OrderedDict` remembers order and can move a key to the end in O(1), so: on a **hit**, move the key to the end; on a **miss**, compute, store at the end, and if the cache is too big, drop the **first** (least recently used) key. ("Implement an LRU cache" is a very common interview question.)

In [39]:
def lru_memoize(maxsize=128):
    def decorator(func):
        cache = OrderedDict()
        stats = {"hits": 0, "misses": 0}

        @functools.wraps(func)
        def wrapper(*args):                            # positional, hashable args only (keeps the idea clear)
            if args in cache:
                stats["hits"] += 1
                cache.move_to_end(args)                # mark as most recently used
                return cache[args]
            stats["misses"] += 1
            result = func(*args)
            cache[args] = result
            if len(cache) > maxsize:
                cache.popitem(last=False)              # evict the least recently used
            return result

        wrapper.cache_info = lambda: (stats["hits"], stats["misses"], len(cache))
        return wrapper
    return decorator


def slow_square(x):
    return x * x


ours = lru_memoize(maxsize=3)(slow_square)
reference = functools.lru_cache(maxsize=3)(slow_square)
call_sequence = [random.randint(0, 6) for _ in range(500)]
for x in call_sequence:
    assert ours(x) == reference(x)
info = reference.cache_info()
assert ours.cache_info() == (info.hits, info.misses, info.currsize), (ours.cache_info(), info)
print(f"✅ lru_memoize matches functools.lru_cache exactly: hits={info.hits}, misses={info.misses}, size={info.currsize}")
print("   name preserved by functools.wraps:", ours.__name__)

✅ lru_memoize matches functools.lru_cache exactly: hits=206, misses=294, size=3
   name preserved by functools.wraps: slow_square


### 4) A `Timer` context manager

In [40]:
class Timer:
    """Measure how long a `with` block takes; works even if the block raises."""

    def __init__(self, label="block"):
        self.label = label
        self.elapsed = None

    def __enter__(self):
        self._start = time.perf_counter()
        return self                                    # `as t` gives access to t.elapsed afterwards

    def __exit__(self, exc_type, exc, traceback):
        self.elapsed = time.perf_counter() - self._start
        status = "failed" if exc_type else "ok"
        print(f"  ⏱️ {self.label}: {self.elapsed * 1000:.1f} ms ({status})")
        return False                                   # never swallow exceptions


outer_start = time.perf_counter()
with Timer("sleep 50 ms") as t:
    time.sleep(0.05)
outer_elapsed = time.perf_counter() - outer_start
assert 0.05 <= t.elapsed <= outer_elapsed, (t.elapsed, outer_elapsed)   # at least the sleep, at most the outside measurement

failing = Timer("crashing block")
try:
    with failing:
        time.sleep(0.01)
        raise KeyError("oops")
except KeyError:
    pass
assert failing.elapsed is not None and failing.elapsed >= 0.01          # still measured, and the error still propagated
print("✅ Timer measures correctly and lets exceptions propagate")

  ⏱️ sleep 50 ms: 55.5 ms (ok)
  ⏱️ crashing block: 15.0 ms (failed)
✅ Timer measures correctly and lets exceptions propagate


## ⚠️ Common Pitfalls

### ❌ Pitfall 1 — Looping over an iterator twice

In [41]:
clean_scores = map(float, ["1.5", "2.5", "3.0"])
print("❌ first pass :", sum(clean_scores))
print("❌ second pass:", sum(clean_scores), "← silently 0: the map object is exhausted")

clean_scores = list(map(float, ["1.5", "2.5", "3.0"]))     # ✅ materialize once if you need several passes
print("✅ both passes:", sum(clean_scores), sum(clean_scores))

❌ first pass : 7.0
❌ second pass: 0 ← silently 0: the map object is exhausted
✅ both passes: 7.0 7.0


### ❌ Pitfall 2 — `groupby` on unsorted data

In [42]:
logins = ["ana", "ben", "ana", "ana", "ben"]
print("❌", [(user, len(list(group))) for user, group in itertools.groupby(logins)])
print("✅", [(user, len(list(group))) for user, group in itertools.groupby(sorted(logins))], "| or simply:", Counter(logins))

❌ [('ana', 1), ('ben', 1), ('ana', 2), ('ben', 1)]
✅ [('ana', 3), ('ben', 2)] | or simply: Counter({'ana': 3, 'ben': 2})


### ❌ Pitfall 3 — Late-binding closures in a loop

In [43]:
thresholds = [0.3, 0.5, 0.7]
checkers = [lambda p: p >= t for t in thresholds]           # ❌ all use the LAST t (0.7)
print("❌ is 0.6 above each threshold?", [check_fn(0.6) for check_fn in checkers])

checkers = [lambda p, t=t: p >= t for t in thresholds]      # ✅ bind each value now
print("✅ is 0.6 above each threshold?", [check_fn(0.6) for check_fn in checkers])

❌ is 0.6 above each threshold? [False, False, False]
✅ is 0.6 above each threshold? [True, True, False]


### ❌ Pitfall 4 — Reading a `defaultdict` creates keys

In [44]:
inventory = defaultdict(int, {"gpu": 2})
if inventory["tpu"] == 0:                                   # ❌ reading a missing key INSERTS it
    pass
print("❌ keys after a read:", dict(inventory))

inventory = defaultdict(int, {"gpu": 2})
if inventory.get("tpu", 0) == 0 and "tpu" not in inventory: # ✅ .get and `in` don't insert
    pass
print("✅ keys after a read:", dict(inventory))

❌ keys after a read: {'gpu': 2, 'tpu': 0}
✅ keys after a read: {'gpu': 2}


### ❌ Pitfall 5 — `@contextmanager` without `try/finally`

In [45]:
opened_connections = []


@contextmanager
def connection_buggy():
    opened_connections.append("conn")
    yield "conn"
    opened_connections.remove("conn")                       # ❌ skipped when the block raises


@contextmanager
def connection_safe():
    opened_connections.append("conn")
    try:
        yield "conn"
    finally:
        opened_connections.remove("conn")                   # ✅ always runs


for manager in (connection_buggy, connection_safe):
    opened_connections.clear()
    try:
        with manager():
            raise TimeoutError("query timed out")
    except TimeoutError:
        pass
    print(f"{'❌' if opened_connections else '✅'} {manager.__name__}: connections left open = {len(opened_connections)}")

❌ connection_buggy: connections left open = 1
✅ connection_safe: connections left open = 0


### ❌ Pitfall 6 — Mixing naive and aware datetimes

In [46]:
deadline_naive = datetime(2026, 9, 30, 17, 0)               # ❌ "17:00" — but in which time zone?
now_aware = datetime.now(UTC)
try:
    print(now_aware < deadline_naive)
except TypeError as err:
    print("❌ TypeError:", err)

deadline = datetime(2026, 9, 30, 17, 0, tzinfo=ZoneInfo("Europe/London"))   # ✅ say the zone explicitly
print("✅ deadline in UTC:", deadline.astimezone(UTC).isoformat(), "| comparison works:", isinstance(now_aware < deadline, bool))

❌ TypeError: can't compare offset-naive and offset-aware datetimes
✅ deadline in UTC: 2026-09-30T16:00:00+00:00 | comparison works: True


## 🏋️ Practice Exercises

Try each one before opening the solution. Run the cell: ⏳ means not attempted, ✅ means correct.

### 🟢 Exercise 1 — Top words
Return the **2 most common words** in `review` as a list of `(word, count)` tuples.

In [47]:
review = "data beats models data beats hype data wins"
top_two = None  # TODO
check("top_two", top_two, [("data", 3), ("beats", 2)], hint="Counter(review.split()).most_common(2)")

⏳ top_two: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
review = "data beats models data beats hype data wins"
top_two = Counter(review.split()).most_common(2)
check("top_two", top_two, [("data", 3), ("beats", 2)])
```
</details>

### 🟢 Exercise 2 — Leaderboard order
Sort `leaderboard` by accuracy **highest first**, breaking ties **alphabetically**, and return only the names.

In [48]:
leaderboard = [("bert", 0.91), ("gpt", 0.93), ("albert", 0.91), ("t5", 0.88)]
ranked_names = None  # TODO
check("ranked_names", ranked_names, ["gpt", "albert", "bert", "t5"], hint="key=lambda row: (-row[1], row[0])")

⏳ ranked_names: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
leaderboard = [("bert", 0.91), ("gpt", 0.93), ("albert", 0.91), ("t5", 0.88)]
ranked_names = [name for name, _ in sorted(leaderboard, key=lambda row: (-row[1], row[0]))]
check("ranked_names", ranked_names, ["gpt", "albert", "bert", "t5"])
```
</details>

### 🟢 Exercise 3 — Is it sorted?
Complete `is_non_decreasing(values)` in **one line** using `all` and `itertools.pairwise`.

In [49]:
def is_non_decreasing(values):
    # TODO: replace the next line
    return None


sorted_tests = [[1, 2, 2, 5], [3, 1], [], [7]]
sorted_results = [is_non_decreasing(v) for v in sorted_tests]
check("is_non_decreasing", None if None in sorted_results else sorted_results, [True, False, True, True],
      hint="return all(a <= b for a, b in itertools.pairwise(values))")

⏳ is_non_decreasing: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def is_non_decreasing(values):
    return all(a <= b for a, b in itertools.pairwise(values))


sorted_tests = [[1, 2, 2, 5], [3, 1], [], [7]]
sorted_results = [is_non_decreasing(v) for v in sorted_tests]
check("is_non_decreasing", sorted_results, [True, False, True, True])
```
Empty and single-item lists have no pairs, and `all([])` is `True` — exactly the behaviour we want. It also stops at the first out-of-order pair.
</details>

### 🟡 Exercise 4 — Streaming moving average
Write a **generator** `moving_average(values, k)` that yields the average of each window of `k` consecutive values, using a `deque(maxlen=k)`. It must work on any iterable, including generators.

In [50]:
def moving_average(values, k):
    # TODO: replace the next line with a generator
    return None


averages = moving_average(iter([1, 2, 3, 4, 5]), 3)
check("moving_average", None if averages is None else list(averages), [2.0, 3.0, 4.0],
      hint="window = deque(maxlen=k); append each value; once len(window) == k, yield sum(window) / k")

⏳ moving_average: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def moving_average(values, k):
    window = deque(maxlen=k)
    total = 0.0
    for value in values:
        if len(window) == k:
            total -= window[0]          # the value about to fall out of the window
        window.append(value)
        total += value
        if len(window) == k:
            yield total / k


averages = moving_average(iter([1, 2, 3, 4, 5]), 3)
check("moving_average", list(averages), [2.0, 3.0, 4.0])
```
Keeping a running `total` makes each step O(1) instead of O(k). For very long float streams, recompute `sum(window)` occasionally to avoid accumulating rounding error.
</details>

### 🟡 Exercise 5 — A decorator with arguments
Complete `clip_output(low, high)`: a decorator factory whose wrapper **clamps** the function's numeric return value into `[low, high]` and preserves the function's name.

In [51]:
def clip_output(low, high):
    def decorator(func):
        # TODO: replace the next line — return a functools.wraps wrapper that clamps func's result
        return None
    return decorator


@clip_output(0, 1)
def raw_score(x):
    return x / 10


check("clip_output", None if raw_score is None else (raw_score.__name__, [raw_score(x) for x in (-5, 5, 70)]),
      ("raw_score", [0, 0.5, 1]), hint="return max(low, min(high, func(*args, **kwargs)))")

⏳ clip_output: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def clip_output(low, high):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return max(low, min(high, func(*args, **kwargs)))
        return wrapper
    return decorator


@clip_output(0, 1)
def raw_score(x):
    return x / 10


check("clip_output", (raw_score.__name__, [raw_score(x) for x in (-5, 5, 70)]), ("raw_score", [0, 0.5, 1]))
```
</details>

### 🔴 Exercise 6 — Sessionize user events (interview classic)
Each event is `(user, ISO timestamp)`, **not sorted**. A user's new **session** starts when more than `gap_minutes` pass since their previous event (a gap of exactly 30 minutes stays in the same session). Return a dict `user → number of sessions`.

In [52]:
def count_sessions(events, gap_minutes=30):
    # TODO: replace the next line with your implementation
    return None


events = [
    ("ana", "2026-05-01T10:30"), ("ben", "2026-05-01T09:05"), ("ana", "2026-05-01T09:00"),
    ("cy", "2026-05-01T23:50"), ("ana", "2026-05-01T09:20"), ("ben", "2026-05-01T11:00"),
    ("ana", "2026-05-01T10:05"), ("ben", "2026-05-01T11:30"), ("cy", "2026-05-02T00:10"),
]
check("count_sessions", count_sessions(events), {"ana": 2, "ben": 2, "cy": 1},
      hint="Sort by (user, time); groupby user; within a user, count pairwise gaps > timedelta(minutes=gap_minutes), plus 1.")

⏳ count_sessions: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def count_sessions(events, gap_minutes=30):
    parsed = sorted((user, datetime.fromisoformat(ts)) for user, ts in events)   # sort by user, then time
    gap = timedelta(minutes=gap_minutes)
    sessions = {}
    for user, group in itertools.groupby(parsed, key=lambda e: e[0]):
        times = [t for _, t in group]
        sessions[user] = 1 + sum(1 for earlier, later in itertools.pairwise(times) if later - earlier > gap)
    return sessions


events = [
    ("ana", "2026-05-01T10:30"), ("ben", "2026-05-01T09:05"), ("ana", "2026-05-01T09:00"),
    ("cy", "2026-05-01T23:50"), ("ana", "2026-05-01T09:20"), ("ben", "2026-05-01T11:00"),
    ("ana", "2026-05-01T10:05"), ("ben", "2026-05-01T11:30"), ("cy", "2026-05-02T00:10"),
]
check("count_sessions", count_sessions(events), {"ana": 2, "ben": 2, "cy": 1})
```
O(n log n) for the sort. Talk through the edge cases: exactly 30 minutes (same session), sessions crossing midnight (`cy`), and unsorted input. With a stream too big to sort, keep a dict `user → last timestamp` instead (O(n), but it needs events in time order).
</details>

## 🚀 Mini Project: Streaming Apache Log Analyzer

**Goal:** analyze **2,000 real lines from an Apache web server error log** (from [Loghub](https://github.com/logpai/loghub), a public collection of system logs used in log-analysis research) with a streaming, generator-based pipeline — the way you'd process a 50 GB production log.

**Questions:** How many errors are there and which messages dominate? When do errors spike? Is there a burst of errors in a short window? We finish by saving a JSON report.

**Steps:** download and cache → parse lazily → message templates → errors per hour → burst detection → error codes → save the report.

### Step 1 — Download once, cache in `_outputs/`

In [53]:
LOG_URL = "https://raw.githubusercontent.com/logpai/loghub/master/Apache/Apache_2k.log"
log_path = OUTPUT_DIR / "Apache_2k.log"

if log_path.exists():
    print(f"using cached file {log_path}")
else:
    with urllib.request.urlopen(LOG_URL, timeout=30) as response:
        log_path.write_bytes(response.read())
    print(f"downloaded {log_path}")

print(f"{log_path.stat().st_size:,} bytes")
with open(log_path, encoding="utf-8") as f:
    for line in itertools.islice(f, 3):                 # peek without reading the whole file
        print("  ", line.rstrip())

using cached file _outputs/Apache_2k.log
171,239 bytes
   [Sun Dec 04 04:47:44 2005] [notice] workerEnv.init() ok /etc/httpd/conf/workers2.properties
   [Sun Dec 04 04:47:44 2005] [error] mod_jk child workerEnv in error state 6
   [Sun Dec 04 04:51:08 2005] [notice] jk2_init() Found child 6725 in scoreboard slot 10


### Step 2 — Parse lazily with a generator pipeline

Each line looks like `[Sun Dec 04 04:47:44 2005] [error] message`. We parse with plain string methods: `partition` splits once around a separator. Malformed lines are **counted, not silently dropped**.

In [54]:
LogEntry = namedtuple("LogEntry", ["timestamp", "level", "message"])


def read_lines(path):
    with open(path, encoding="utf-8") as f:            # the file is closed when the generator finishes
        for line in f:
            yield line.rstrip("\n")


def parse_entries(lines, problems):
    for line in lines:
        stamp_text, sep1, rest = line.removeprefix("[").partition("] [")
        level, sep2, message = rest.partition("] ")
        if not (sep1 and sep2):
            problems["missing brackets"] += 1
            continue
        try:
            stamp = datetime.strptime(stamp_text, "%a %b %d %H:%M:%S %Y")
        except ValueError:
            problems["bad timestamp"] += 1
            continue
        yield LogEntry(stamp, level, message.strip())


parse_problems = Counter()
entries = list(parse_entries(read_lines(log_path), parse_problems))   # 2k lines is small enough to keep
n_lines = sum(1 for _ in read_lines(log_path))
print(f"lines: {n_lines:,} | parsed entries: {len(entries):,} | problems: {dict(parse_problems) or 'none'}")
print("example:", entries[0])

out_of_order = sum(1 for a, b in itertools.pairwise(entries) if b.timestamp < a.timestamp)
is_chronological = out_of_order == 0
print(f"already in time order: {is_chronological} ({out_of_order} lines have an earlier timestamp than the line before)")
if not is_chronological:
    entries.sort(key=lambda e: e.timestamp)            # groupby by time (step 4) needs sorted input
first_seen, last_seen = entries[0].timestamp, entries[-1].timestamp
print(f"covers {first_seen} → {last_seen} ({last_seen - first_seen}) — server local time; the log doesn't say which zone")

lines: 2,000 | parsed entries: 2,000 | problems: none
example: LogEntry(timestamp=datetime.datetime(2005, 12, 4, 4, 47, 44), level='notice', message='workerEnv.init() ok /etc/httpd/conf/workers2.properties')
already in time order: False (33 lines have an earlier timestamp than the line before)
covers 2005-12-04 04:47:44 → 2005-12-05 19:15:57 (1 day, 14:28:13) — server local time; the log doesn't say which zone


### Step 3 — Levels and message templates

Log messages differ only in numbers (process ids, slots, IPs). Replacing every run of digits with `<N>` turns thousands of lines into a few **templates**. `itertools.groupby` over the characters, keyed by `str.isdigit`, finds those runs.

In [55]:
def to_template(message):
    return "".join("<N>" if is_digit else "".join(chars)
                   for is_digit, chars in itertools.groupby(message, key=str.isdigit))


print(to_template("jk2_init() Found child 6725 in scoreboard slot 10"))

level_counts = Counter(e.level for e in entries)
print("\nlevels:", level_counts.most_common())

template_counts = Counter((e.level, to_template(e.message)) for e in entries)
print(f"\n{len(template_counts)} templates cover all {len(entries):,} entries:")
for (level, template), count in template_counts.most_common():
    print(f"  {count:>5}  [{level:<6}] {template}")

error_share = level_counts["error"] / len(entries)
top_error_template, top_error_count = max(((t, c) for (lvl, t), c in template_counts.items() if lvl == "error"), key=lambda tc: tc[1])
print(f"\n→ {error_share:.1%} of entries are errors; the most common error template accounts for "
      f"{top_error_count / level_counts['error']:.1%} of them: {top_error_template!r}")

jk<N>_init() Found child <N> in scoreboard slot <N>

levels: [('notice', 1405), ('error', 595)]

6 templates cover all 2,000 entries:
    836  [notice] jk<N>_init() Found child <N> in scoreboard slot <N>
    569  [notice] workerEnv.init() ok /etc/httpd/conf/workers<N>.properties
    539  [error ] mod_jk child workerEnv in error state <N>
     32  [error ] [client <N>.<N>.<N>.<N>] Directory index forbidden by rule: /var/www/html/
     12  [error ] jk<N>_init() Can't find child <N> in scoreboard
     12  [error ] mod_jk child init <N> -<N>

→ 29.8% of entries are errors; the most common error template accounts for 90.6% of them: 'mod_jk child workerEnv in error state <N>'


### Step 4 — Errors per hour with `groupby`

In [56]:
errors = [e for e in entries if e.level == "error"]
hour_key = lambda e: e.timestamp.replace(minute=0, second=0)       # truncate to the hour

errors_per_hour = [(hour, sum(1 for _ in group)) for hour, group in itertools.groupby(errors, key=hour_key)]
peak_count = max(count for _, count in errors_per_hour)
for hour, count in errors_per_hour:
    print(f"{hour:%a %d %b %H:00}  {count:>4}  {'█' * math.ceil(count / peak_count * 40)}")

peak_hour, _ = max(errors_per_hour, key=lambda hc: hc[1])
print(f"\n→ peak: {peak_count} errors in the hour starting {peak_hour:%a %d %b %H:00} "
      f"({peak_count / len(errors):.1%} of all errors in one hour)")

Sun 04 Dec 04:00    26  ████████████
Sun 04 Dec 05:00    16  ████████
Sun 04 Dec 06:00    90  ████████████████████████████████████████
Sun 04 Dec 07:00    28  █████████████
Sun 04 Dec 08:00     1  █
Sun 04 Dec 09:00     1  █
Sun 04 Dec 10:00     1  █
Sun 04 Dec 11:00     3  ██
Sun 04 Dec 12:00     1  █
Sun 04 Dec 13:00     1  █
Sun 04 Dec 14:00     1  █
Sun 04 Dec 15:00     2  █
Sun 04 Dec 16:00    27  ████████████
Sun 04 Dec 17:00    37  █████████████████
Sun 04 Dec 18:00     1  █
Sun 04 Dec 19:00    29  █████████████
Sun 04 Dec 20:00    46  █████████████████████
Mon 05 Dec 01:00     2  █
Mon 05 Dec 03:00    23  ███████████
Mon 05 Dec 04:00    13  ██████
Mon 05 Dec 05:00     7  ████
Mon 05 Dec 06:00     3  ██
Mon 05 Dec 07:00    44  ████████████████████
Mon 05 Dec 09:00     4  ██
Mon 05 Dec 10:00    45  ████████████████████
Mon 05 Dec 11:00    11  █████
Mon 05 Dec 12:00     9  ████
Mon 05 Dec 13:00    45  ████████████████████
Mon 05 Dec 14:00     5  ███
Mon 05 Dec 15:00    11  █████
M

### Step 5 — Burst detection with a sliding window (`deque`)

What is the largest number of errors inside **any 60-second window**? A `deque` holds the timestamps inside the current window: append the newest, pop old ones from the left. Each timestamp enters and leaves once, so this is O(n).

In [57]:
def max_events_in_window(timestamps, window):
    in_window = deque()
    best_count, best_end = 0, None
    for ts in timestamps:
        in_window.append(ts)
        while ts - in_window[0] > window:
            in_window.popleft()
        if len(in_window) > best_count:
            best_count, best_end = len(in_window), ts
    return best_count, best_end


window = timedelta(seconds=60)
burst_count, burst_end = max_events_in_window((e.timestamp for e in errors), window)

brute_force = max(sum(1 for other in errors if timedelta(0) <= e.timestamp - other.timestamp <= window) for e in errors)
assert burst_count == brute_force                                   # verify the O(n) version against an O(n²) one
print(f"→ worst burst: {burst_count} errors within 60 seconds, ending at {burst_end} (verified by brute force)")

→ worst burst: 11 errors within 60 seconds, ending at 2005-12-04 20:47:17 (verified by brute force)


### Step 6 — Which error states and child processes show up?

In [58]:
error_states = Counter()
children_found, children_missing = set(), set()
for e in entries:
    words = e.message.split()
    if e.message.startswith("mod_jk child workerEnv in error state"):
        error_states[int(words[-1])] += 1
    elif e.message.startswith("jk2_init() Found child"):
        children_found.add(int(words[3]))
    elif e.message.startswith("jk2_init() Can't find child"):
        children_missing.add(int(words[4]))

print("workerEnv error states:", error_states.most_common())
print(f"child processes seen in the scoreboard: {len(children_found):,} | reported missing: {len(children_missing)}")
print("missing children that were ALSO seen at some point:", len(children_missing & children_found))

forbidden_clients = Counter(e.message.split("]")[0].removeprefix("[client ")
                            for e in entries if "Directory index forbidden" in e.message)
print(f"'Directory index forbidden' came from {len(forbidden_clients)} distinct client IPs "
      f"({sum(forbidden_clients.values())} requests); max from one IP: {max(forbidden_clients.values())}")

workerEnv error states: [(6, 369), (7, 101), (8, 44), (9, 20), (10, 5)]
child processes seen in the scoreboard: 830 | reported missing: 12
missing children that were ALSO seen at some point: 0
'Directory index forbidden' came from 32 distinct client IPs (32 requests); max from one IP: 1


### Step 7 — Save a JSON report and verify the round trip

In [59]:
report = {
    "source": LOG_URL,
    "entries": len(entries),
    "first_seen": first_seen.isoformat(),
    "last_seen": last_seen.isoformat(),
    "levels": dict(level_counts),
    "top_templates": [{"level": lvl, "template": t, "count": c} for (lvl, t), c in template_counts.most_common(5)],
    "peak_error_hour": {"hour": peak_hour.isoformat(), "errors": peak_count},
    "worst_60s_burst": {"errors": burst_count, "ending_at": burst_end.isoformat()},
    "error_states": {str(state): n for state, n in error_states.items()},          # JSON keys must be strings
}
report_path = OUTPUT_DIR / "apache_log_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

reloaded = json.loads(report_path.read_text(encoding="utf-8"))
assert reloaded == report
print(f"✅ wrote {report_path} ({report_path.stat().st_size:,} bytes) and it round-trips exactly")
print(json.dumps({k: reloaded[k] for k in ["entries", "levels", "peak_error_hour", "worst_60s_burst"]}, indent=2))

✅ wrote _outputs/apache_log_report.json (1,186 bytes) and it round-trips exactly
{
  "entries": 2000,
  "levels": {
    "notice": 1405,
    "error": 595
  },
  "peak_error_hour": {
    "hour": "2005-12-04T06:00:00",
    "errors": 90
  },
  "worst_60s_burst": {
    "errors": 11,
    "ending_at": "2005-12-04T20:47:17"
  }
}


**Stretch goals**
1. Make the pipeline **fully streaming**: compute level counts, templates, and the burst detector in one pass over `parse_entries(read_lines(...))` without building the `entries` list, and measure peak memory with `tracemalloc`.
2. Download another Loghub sample (for example `HDFS/HDFS_2k.log` or `Linux/Linux_2k.log`), write a new `parse_entries` for its format, and reuse every later step unchanged.
3. Wrap `read_lines` in the `retry` decorator from section 7 so a flaky network download is retried with exponential backoff.

### 🗣️ How to talk about this in an interview
- "I built a lazy generator pipeline — read lines → parse into namedtuples → aggregate — so memory stays flat no matter how big the log is; malformed lines are counted, not silently dropped."
- "I collapsed messages into templates by replacing digit runs with a placeholder, which reduced 2,000 lines to a handful of patterns and showed that one `workerEnv ... error state <N>` template accounts for the vast majority of errors."
- "Per-hour counts use `itertools.groupby`, which only groups consecutive items — so I checked the time order first, found out-of-order lines, and sorted before grouping instead of trusting the file."
- "For bursts I used a sliding window with a `deque` — O(n) — and verified it against an O(n²) brute force."
- "Timestamps in this log are naive local time, so I didn't pretend to know the zone; in production I'd log in UTC with ISO 8601."

## 🎤 Interview Q&A

Try answering **out loud** before opening each answer.

### 🧠 Concepts

**Q1. What is the difference between an iterable, an iterator, and a generator?**

<details><summary>Show answer</summary>

- **30-second answer:** An **iterable** can be looped over and returns an iterator from `iter()` (lists, dicts, strings). An **iterator** returns the next item from `next()`, remembers its position, and raises `StopIteration` when done — it's one-shot. A **generator** is the easiest way to write an iterator: a function with `yield` (or a generator expression).
- **Go deeper:** Iterators implement `__iter__` (returning `self`) and `__next__`. Generators also support `send()`, `throw()`, and `close()`, and `yield from` delegates to a sub-iterator. PEP 479 turns a `StopIteration` escaping a generator body into `RuntimeError`.
- **❌ Common wrong answer:** "A list is an iterator" — `iter(lst) is lst` is `False`; you can loop over a list many times because each loop gets a fresh iterator.

</details>

**Q2. When would you use a generator instead of a list?**

<details><summary>Show answer</summary>

- **30-second answer:** When the data is large or infinite, when you only need one pass, or when you may stop early. A generator holds one item at a time, so memory is constant — in this notebook a generator used thousands of times less peak memory than a list for summing 1M squares.
- **Go deeper:** Generators compose into pipelines (read → parse → filter) and give *lazy evaluation*: work happens only when a consumer pulls. Use a list when you need `len`, indexing, several passes, or sorting.
- **❌ Common wrong answer:** "Generators are always faster." They save memory; per-item speed is similar or slightly slower.

</details>

**Q3. What is a closure, and what is late binding?**

<details><summary>Show answer</summary>

- **30-second answer:** A closure is an inner function that captures variables from its enclosing function and keeps them alive after the outer function returns (e.g. `make_multiplier(3)`). Late binding means the captured variable is looked up **when the inner function is called**, so lambdas created in a loop all see the loop variable's final value.
- **Go deeper:** Captured variables live in cells (`func.__closure__`). Use `nonlocal` to assign to them. Fix late binding with a default argument (`lambda i=i: i`) or `functools.partial`. Closures are the mechanism behind decorators.
- **❌ Common wrong answer:** "The closure copies the value at creation time."

</details>

**Q4. What is a decorator, and why use `functools.wraps`?**

<details><summary>Show answer</summary>

- **30-second answer:** A decorator is a callable that takes a function and returns a replacement, usually a wrapper adding behaviour (timing, logging, retries, caching, auth). `@deco` above `def f` means `f = deco(f)`. `functools.wraps` copies `__name__`, `__doc__`, `__module__`, etc. and sets `__wrapped__`, so logs, `help()`, debuggers, and frameworks see the original function.
- **Go deeper:** Decorators with arguments add a factory layer: `retry(times=3)` returns the decorator. Stacked decorators apply bottom-up. Frameworks like FastAPI and pytest inspect signatures, which is another reason metadata must be preserved.
- **❌ Common wrong answer:** "Decorators modify the original function's code" — they wrap or replace it; the original is untouched.

</details>

**Q5. How does the `with` statement work? How do you write your own context manager?**

<details><summary>Show answer</summary>

- **30-second answer:** `with` calls `__enter__` (its return value goes to `as`), runs the block, and always calls `__exit__(exc_type, exc, tb)` — on success or error. Write one as a class with those two methods, or as a generator with `@contextlib.contextmanager` where code before `yield` is setup and cleanup sits in `finally`.
- **Go deeper:** If `__exit__` returns a truthy value the exception is suppressed (that's how `contextlib.suppress` works). Without `try/finally` around `yield`, an exception in the block skips your cleanup. Use cases: files, locks, DB transactions, temporary settings, `torch.no_grad()`.
- **❌ Common wrong answer:** "`with` just closes files" — it's a general protocol for any setup/teardown.

</details>

**Q6. How does `functools.lru_cache` work, and when is it a bad idea?**

<details><summary>Show answer</summary>

- **30-second answer:** It stores results in a dict keyed by the call arguments; a repeat call returns the stored result. With a `maxsize`, the least recently used entry is evicted. It turned exponential naive Fibonacci into linear time here.
- **Go deeper:** Arguments must be hashable. It's wrong for functions with side effects or that depend on changing external state (stale results), risky for huge return values (memory), and on methods it keeps `self` alive. Inspect with `cache_info()`, reset with `cache_clear()`. `functools.cache` = unbounded `lru_cache`.
- **❌ Common wrong answer:** "It caches to disk" or "it works with any arguments, including lists."

</details>

**Q7. `dict` vs `defaultdict` vs `Counter`; `list` vs `deque` — when do you use each?**

<details><summary>Show answer</summary>

- **30-second answer:** `defaultdict(factory)` auto-creates missing keys — perfect for grouping. `Counter` is a dict subclass for counting with `most_common`. `deque` gives O(1) appends and pops at both ends (queues, BFS, sliding windows with `maxlen`), whereas `list.pop(0)`/`insert(0)` are O(n).
- **Go deeper:** Reading a missing key on a `defaultdict` inserts it — use `.get` or `in` to test. `Counter` returns 0 for missing keys without inserting. `deque` indexing in the middle is O(n), so it isn't a list replacement for random access.
- **❌ Common wrong answer:** "`deque` is just a faster list" — it's faster only at the ends.

</details>

**Q8. What's the difference between naive and aware datetimes? How should you store timestamps?**

<details><summary>Show answer</summary>

- **30-second answer:** Naive datetimes have no time zone; aware ones have `tzinfo`. Comparing the two raises `TypeError`. Store timestamps as aware UTC (ISO 8601 or epoch), and convert to local zones with `zoneinfo` only for display.
- **Go deeper:** Arithmetic between aware datetimes in the *same* zone is wall-clock arithmetic — across a DST change, `after - before` can say 1 day while only 23 hours really passed; convert to UTC for true durations. `datetime.utcnow()` returns a *naive* value and is deprecated since Python 3.12 — use `datetime.now(UTC)`.
- **❌ Common wrong answer:** "Store local time plus the zone name and it's fine" — ambiguous during DST fall-back hours, and painful to compare across regions.

</details>

### 💻 Coding

**Q9. Write a `retry` decorator that takes the number of attempts and the exceptions to retry on.**

<details><summary>Show answer</summary>

- **30-second answer:** Three layers: `def retry(times, exceptions): def decorator(func): @wraps(func) def wrapper(*args, **kwargs): for attempt in range(1, times + 1): try: return func(*args, **kwargs) except exceptions: if attempt == times: raise` … `return wrapper` / `return decorator`.
- **Go deeper:** Add exponential backoff with jitter (`delay * 2**attempt * random.uniform(0.5, 1.5)`), retry only transient errors (timeouts, 429/503 — not 400s), log each attempt, and make the operation idempotent. Libraries like `tenacity` do this in production.
- **❌ Common wrong answer:** Catching `Exception` for everything (retries bugs forever), or forgetting `return` so the wrapper returns `None`.

</details>

**Q10. Implement an LRU cache with O(1) `get` and `put`.**

<details><summary>Show answer</summary>

- **30-second answer:** Use an `OrderedDict`: `get` → if the key exists, `move_to_end(key)` and return the value; `put` → set the value, `move_to_end(key)`, and if `len > capacity`, `popitem(last=False)` to evict the least recently used.
- **Go deeper:** Without `OrderedDict`, combine a dict (key → node) with a doubly linked list (recency order) — that's what interviewers want if they forbid the library. A plain `dict` also keeps insertion order, but moving a key to the end means `pop` + reinsert.
- **❌ Common wrong answer:** Storing keys in a list and calling `list.remove(key)` on every access — that's O(n).

</details>

**Q11. Count the ERROR lines per hour in a 50 GB log file on a laptop.**

<details><summary>Show answer</summary>

- **30-second answer:** Stream it: `with open(path) as f: for line in f:` → parse only the timestamp and level → `counts[hour] += 1` in a `Counter`. Memory is O(number of hours), not O(file size).
- **Go deeper:** Filter cheaply before parsing (`if "[error]" in line`), use generator stages for readability, and parallelize by splitting the file into byte ranges (or use tools like DuckDB/Polars/Spark). Don't use `groupby` unless the file is guaranteed to be time-ordered.
- **❌ Common wrong answer:** `f.readlines()` or `pandas.read_csv` of the whole file into memory.

</details>

### 🐛 Debugging Scenarios

**Q12. A function computes `total = sum(rows)` and then `count = len(list(rows))`, but `count` is always 0. What's wrong?**

<details><summary>Show answer</summary>

- **30-second answer:** `rows` is an iterator or generator (e.g. from `map`, a file, or a generator function); `sum` consumed it, so the second pass is empty. Materialize it once with `rows = list(rows)` or compute both in a single loop.
- **Go deeper:** `itertools.tee` can split an iterator, but it buffers items in memory, so for large data a single pass that updates both `total` and `count` is best.
- **❌ Common wrong answer:** "`len` doesn't work on lists" or "there must be no rows."

</details>

**Q13. A report built with `itertools.groupby` lists the same category several times. Why?**

<details><summary>Show answer</summary>

- **30-second answer:** `groupby` only groups **consecutive** items with the same key. Sort by the same key first (`sorted(data, key=k)`) or use a `defaultdict(list)`/`Counter` when you don't need sorted output.
- **Go deeper:** Also note each group is a lazy iterator tied to the main `groupby` iterator — convert it (`list(group)`) before advancing to the next key, or it will be empty.
- **❌ Common wrong answer:** "`groupby` is like SQL `GROUP BY`" — SQL groups globally; `itertools.groupby` works like Unix `uniq`.

</details>

**Q14. Code converts `json.loads(json.dumps(stats))` and then `stats[1]` raises `KeyError`, even though key `1` existed. Why?**

<details><summary>Show answer</summary>

- **30-second answer:** JSON object keys are always strings, so `{1: ...}` comes back as `{"1": ...}`. Convert keys back (`{int(k): v for k, v in d.items()}`) or store data as a list of records.
- **Go deeper:** Other lossy round trips: tuples become lists, sets and datetimes aren't serializable at all (convert with `list()` / `isoformat()` or pass `default=`), and floats like `nan` produce non-standard JSON unless `allow_nan=False` is set to reject them.
- **❌ Common wrong answer:** "The JSON file got corrupted."

</details>

## 🧪 Quick Quiz

Predict the output, then reveal. (Every answer below was produced by running the code.)

**1.** `print(all([]), any([]))`
<details><summary>Answer</summary>

`True False` — "every element of an empty list is truthy" is vacuously true; "at least one is" is false.
</details>

**2.** `[key for key, _ in itertools.groupby("AABBA")]`
<details><summary>Answer</summary>

`['A', 'B', 'A']` — groupby only merges consecutive equal items.
</details>

**3.** `fs = [lambda: i for i in range(3)]; print([f() for f in fs])`
<details><summary>Answer</summary>

`[2, 2, 2]` — late binding: every lambda reads `i` when called, after the loop finished.
</details>

**4.** `m = map(str.upper, "ab"); print(list(m), list(m))`
<details><summary>Answer</summary>

`['A', 'B'] []` — a map object is a one-shot iterator.
</details>

**5.** `print((date(2025, 1, 1) - date(2024, 1, 1)).days)`
<details><summary>Answer</summary>

`366` — 2024 is a leap year.
</details>

## 📚 Resources

### 📖 Official Docs
- [Built-in Functions](https://docs.python.org/3/library/functions.html) — every built-in, with exact behaviour
- [itertools — Functions creating iterators for efficient looping](https://docs.python.org/3/library/itertools.html) — read the "recipes" at the bottom
- [functools — Higher-order functions and operations on callable objects](https://docs.python.org/3/library/functools.html)
- [collections — Container datatypes](https://docs.python.org/3/library/collections.html)
- [contextlib — Utilities for with-statement contexts](https://docs.python.org/3/library/contextlib.html)
- [zoneinfo — IANA time zone support](https://docs.python.org/3/library/zoneinfo.html) and [datetime — Basic date and time types](https://docs.python.org/3/library/datetime.html)
- [Functional Programming HOWTO](https://docs.python.org/3/howto/functional.html) — iterators, generators, and itertools explained together

### 🎥 Videos
- [Corey Schafer — Python Tutorial: Decorators - Dynamically Alter The Functionality Of Your Functions](https://www.youtube.com/watch?v=FsAPt_9Bf3U) (30 min) — builds from closures to decorators step by step, exactly like section 7
- [Corey Schafer — Python Tutorial: Generators - How to use them and the benefits you receive](https://www.youtube.com/watch?v=bD05uGo_sVI) (11 min) — short, clear demo of `yield` and the memory benefit
- [PyData — James Powell: So you want to be a Python expert? | PyData Seattle 2017](https://www.youtube.com/watch?v=cKPlPJyQrt4) (1 h 54 min) — a famous deep dive tying together decorators, generators, and context managers; watch after finishing this notebook

### 📄 PEPs (the design documents behind these features)
- [PEP 255 – Simple Generators](https://peps.python.org/pep-0255/) — why `yield` was added
- [PEP 318 – Decorators for Functions and Methods](https://peps.python.org/pep-0318/) — the `@` syntax and its motivation
- [PEP 343 – The "with" Statement](https://peps.python.org/pep-0343/) — the context manager protocol
- [PEP 615 – Support for the IANA Time Zone Database in the Standard Library](https://peps.python.org/pep-0615/) — where `zoneinfo` came from

### 📘 Books & Courses
- [Python 3 Module of the Week](https://pymotw.com/3/) — Doug Hellmann's example-driven tour of the standard library
- [Beyond the Basic Stuff with Python](https://inventwithpython.com/beyond/) — Al Sweigart, free online; idiomatic Python and common gotchas
- [Real Python — Primer on Python Decorators](https://realpython.com/primer-on-python-decorators/) and [How to Use Generators and yield in Python](https://realpython.com/introduction-to-python-generators/)

### 🏋️ Practice
- [HackerRank — Python domain](https://www.hackerrank.com/domains/python) — includes dedicated Collections, Itertools, Date and Time, and Closures & Decorators tracks
- [Loghub](https://github.com/logpai/loghub) — more real log files (HDFS, Linux, Spark, …) to extend the mini project

## 📝 Summary Cheat Sheet

| Concept | What it does | Key API / rule |
|---|---|---|
| Everyday built-ins | size, order, test, pair up | `sorted(key=, reverse=)` (stable), `min/max(key=, default=)`, `any/all`, `zip`, `enumerate` |
| map / filter / lambda | transform & select lazily | `map(int, parts)`; comprehension when a lambda is needed |
| Iterators | one-shot position-keeping streams | `iter()`, `next()`, `StopIteration`; iterate twice → empty |
| Generators | lazy values with `yield` | `yield`, `yield from`, `(x for x in ...)`, `itertools.islice` |
| functools | work with functions | `partial`, `reduce`, `lru_cache(maxsize)` / `cache`, `wraps` |
| Closures | functions that remember | inner function + `nonlocal`; late binding → `lambda i=i:` |
| Decorators | wrap behaviour around functions | `@deco` ≡ `f = deco(f)`; args → factory → decorator → wrapper |
| Context managers | guaranteed cleanup | `__enter__/__exit__`; `@contextmanager` + `try/finally`; `suppress`, `chdir` |
| collections | specialised containers | `Counter.most_common`, `defaultdict(list)`, `deque(maxlen=)`, `namedtuple` |
| itertools | iterator algebra | `chain`, `groupby` (sort first!), `product`, `combinations`, `pairwise`, `batched` |
| datetime & zoneinfo | time done right | `datetime.now(UTC)`, `astimezone(ZoneInfo(...))`, `fromisoformat`, `timedelta` |
| json | text interchange | `dumps/loads(indent=)`; keys → str, tuples → lists, datetimes → `isoformat()` |
| os / sys / pathlib | environment & files | `os.environ.get`, `sys.version_info`, `Path / "x"`, `mkdir(parents=True)`, `rglob` |

## ➡️ What's Next

**[03 · Object-Oriented Programming in Python](03_OOP_in_Python.ipynb)** — you've been *using* objects with special methods (`__enter__`, `__next__`, `__iter__`); next you'll design your own classes, dataclasses, and protocols, which is how libraries like scikit-learn and PyTorch structure models.